# Lab 08 — 2 つの Data Agent を作成・公開

同じ業務定義（指示）で、データソースだけが異なる 2 つの Data Agent を作成します。

| Agent | 接続先 |
|---|---|
| `taxi_metrics_agent` | セマンティックモデル `sm_taxi` |
| `taxi_ontology_agent` | Ontology `onto_taxi`（コンテキストソース） |

設定内容はリポジトリの `config/agents.json` から `scripts/lab.py` が埋め込みます。何度実行しても同じ構成になります。

In [ ]:
%pip install -q "fabric-data-agent-sdk==0.1.36a0"

In [ ]:
import json
from datetime import datetime, timezone

def save_result(name, payload):
    """Persist a run summary that scripts/lab.py reads back from OneLake."""
    payload = {"notebook": name, "finishedAtUtc": datetime.now(timezone.utc).isoformat(timespec="seconds"), **payload}
    notebookutils.fs.put(f"Files/lab_results/{name}.json", json.dumps(payload, ensure_ascii=False, default=str), True)
    print(json.dumps(payload, ensure_ascii=False, indent=2, default=str))

checks = []

def check(name, expected, actual, tolerance=None):
    if tolerance is None:
        passed = expected == actual
    else:
        passed = actual is not None and abs(float(expected) - float(actual)) <= tolerance
    checks.append({"check": name, "expected": expected, "actual": actual, "passed": bool(passed)})

In [ ]:
import base64
import json
import time
import warnings

from fabric.dataagent.client import create_data_agent

AGENTS = json.loads(base64.b64decode("{{AGENT_CONFIG_B64}}").decode("utf-8"))
warnings.filterwarnings("ignore", category=FutureWarning)

def select_all(datasource, root_id=None):
    """Select every element (tables, columns, measures) so the agent can query them."""
    token = None
    while True:
        page = datasource.get_elements(root_id=root_id, continuation_token=token)
        for element in page.get("value", []):
            if element.get("isSelected") is False:
                datasource.update_element(element["id"], is_selected=True)
            if element.get("hasChildren") or element.get("children"):
                select_all(datasource, element["id"])
        token = page.get("continuationToken")
        if not token:
            break

In [ ]:
summary = {}

def find_source(list_sources, name, attempts=6, wait_seconds=10):
    """A just-added source can take a few seconds to appear in the list (eventual consistency)."""
    for attempt in range(attempts):
        found = [s for s in list_sources() if s.get_configuration().get("displayName") == name]
        if found:
            return found[0]
        time.sleep(wait_seconds)
    raise RuntimeError(f"data source {name} did not appear after {attempts} attempts")

def build_agent(config, attach):
    try:
        agent = create_data_agent(config["name"])
        agent.update_settings(ai_instructions="\n".join(config["instructions"]))
        attach(agent, config)
        agent.publish_staging(description=config["description"])
        summary[config["name"]] = {"source": config["source"], "published": True}
    except Exception as error:  # keep going so the other agent is still created
        summary[config["name"]] = {"source": config["source"], "published": False,
                                   "error": f"{type(error).__name__}: {error}"[:500]}

def attach_semantic_model(agent, config):
    name = config["source"]["item"]
    if not [s for s in agent.list_datasources() if s.get_configuration().get("displayName") == name]:
        agent.add_staging_datasource(name, type=config["source"]["type"])
    datasource = find_source(agent.list_datasources, name)
    select_all(datasource)
    datasource.update_configuration(description="NYC green taxi trips April 2026: fact_trips with vendor, payment, trip type, pickup and dropoff zone dimensions.")

build_agent(AGENTS["metricsAgent"], attach_semantic_model)

In [ ]:
def attach_ontology(agent, config):
    name = config["source"]["item"]
    attached = [o for o in agent.list_ontologies() if o.get_configuration().get("displayName") == name]
    ontology = attached[0] if attached else agent.add_staging_ontology(name)
    ontology.select_entities()

build_agent(AGENTS["ontologyAgent"], attach_ontology)

save_result("nb_08_create_agents", {"passed": all(a["published"] for a in summary.values()), "agents": summary,
                                     "next": "scripts/lab.py agent-compare で 2 つの Agent に同じ質問をして比較します。"})